# Parcial 03 - Inteligencia Artificial (EAFIT 2026)

| Campo | Integrante 1 | Integrante 2 | Integrante 3 |
|---|---|---|---|
| Nombre completo |  |  |  |
| ID EAFIT |  |  |  |
| Grupo (Mie/Jue/Vie) |  |  |  |

Fecha de entrega: 29/04/2026  \
Hora: ________

# Seccion A - Diagnostico Situado (20 pts)

## A.1 Autopsia de un clasificador medico de neumonia

In [ ]:
import numpy as np
import pandas as pd

# Matriz de confusion dada (filas = clase real, columnas = prediccion)
classes = ['Normal', 'Bacteriana', 'Viral']
cm = np.array([
    [284,  36,   0],
    [ 12, 268,  40],
    [  4, 116, 200]
])

df_cm = pd.DataFrame(cm, index=[f'Real_{c}' for c in classes], columns=[f'Pred_{c}' for c in classes])
display(df_cm)

total = cm.sum()
acc = np.trace(cm) / total
print(f'Accuracy global: {acc:.4f}')

# Metricas por clase (one-vs-rest)
rows = []
for i, c in enumerate(classes):
    tp = cm[i, i]
    fn = cm[i, :].sum() - tp
    fp = cm[:, i].sum() - tp
    tn = total - tp - fn - fp
    precision = tp / (tp + fp)
    recall = tp / (tp + fn)
    f1 = 2 * precision * recall / (precision + recall)
    rows.append([c, tp, fp, fn, tn, precision, recall, f1])

df_metrics = pd.DataFrame(rows, columns=['Clase', 'TP', 'FP', 'FN', 'TN', 'Precision', 'Recall', 'F1'])
display(df_metrics)

# Error clinico critico mencionado: real bacteriana predicha como normal
danger_count = cm[1, 0]
danger_rate_bacterial = danger_count / cm[1, :].sum()

pred_normal_total = cm[:, 0].sum()
sick_as_normal = cm[1, 0] + cm[2, 0]

print(f'Bacteriana -> Normal: {danger_count} casos ({danger_rate_bacterial:.2%} de las bacterianas)')
print(f'Total predichas como Normal: {pred_normal_total}')
print(f'Enfermos clasificados como Normal: {sick_as_normal} ({sick_as_normal/pred_normal_total:.2%} de los 
 predichos)')

## A.1.a (5 pts)

El patron observado en las curvas corresponde a **overfitting**.

- La accuracy de entrenamiento sigue subiendo casi monotonicamente hasta valores cercanos a 0.99.
- La accuracy de validacion mejora al inicio, pero desde aproximadamente la **epoca 12-15** deja de mejorar y luego cae/oscila.
- De forma consistente, la perdida de entrenamiento sigue bajando, mientras la de validacion deja de bajar y luego empeora.

Esto sugiere que el modelo esta memorizando patrones especificos del set de entrenamiento y perdiendo capacidad de generalizacion.

Causa mas probable en este contexto:
- Dataset relativamente pequeno para fine-tuning profundo (4,800 imagenes) en un problema clinico con variabilidad alta.
- Posible sesgo de dominio (equipos de rayos X, protocolos de adquisicion, marcadores de imagen, diferencias por institucion).
- Capacidad alta de MobileNetV2 respecto al volumen/variabilidad util de datos sin suficiente regularizacion fuerte orientada al dominio.

## A.1.b (7 pts)

Para triaje de emergencia, este modelo **no es clinicamente aceptable como sistema autonomo de descarte**.

Argumentos numericos:
- Accuracy global: **0.783** (insuficiente para una decision de alto riesgo clinico).
- En la clase bacteriana, recall ~ **0.84**: de 320 bacterianas reales, detecta 268 y falla 52.
- Error critico explicitado: **Bacteriana -> Normal = 12 casos** (12/320 = **3.75%** de bacterianas enviadas potencialmente a casa).
- Para la clase viral, recall = **0.63**: pierde 37% de casos virales.

El error mas costoso en este dominio es el **falso negativo de neumonia** (especialmente cuando termina en etiqueta 
), porque implica subtratamiento o no tratamiento oportuno.

Conclusion operacional:
- Podria usarse como apoyo de segunda lectura o priorizacion,
- pero no como filtro final de alta sin revision clinica humana.

## A.1.c (8 pts)

Entrenar mas epocas, por si solo, **probablemente no solucionara** el bajo recall en Viral.

Evidencia desde las curvas:
- Ya hay separacion train/val desde ~epoca 12-15 (sobreajuste).
- Continuar entrenamiento incrementa ajuste al train, no la generalizacion.

### Tres intervenciones tecnicas (ordenadas por impacto esperado)

1. **Rebalanceo dirigido a la clase Viral (class weights + focal loss o hard example mining).**
   - Mecanismo: penaliza mas los errores en ejemplos virales dificiles, desplazando el gradiente para mejorar sensibilidad en esa clase.

2. **Data pipeline clinico robusto (lung segmentation + augmentation especifica de RX torax).**
   - Mecanismo: elimina atajos espurios (texto, bordes, artefactos) y fuerza al modelo a aprender patrones parenquimatosos relevantes para viral.

3. **Ajuste de decision por clase (calibracion + threshold tuning one-vs-rest para Viral).**
   - Mecanismo: no cambia pesos, pero mueve el punto de operacion para reducir FN virales a costa controlada de FP, adecuado para triaje.

Complemento recomendado: early stopping con criterio en metrica de validacion orientada al riesgo clinico (por ejemplo recall minimo por clase o macro-F1 ponderado por costo).

## Pendientes de la Seccion A

- Revisar y ajustar redaccion final al estilo del grupo.
- Si el profe pide mas detalle matematico, agregar mini-tabla TP/FP/FN/TN por clase en markdown.
- Pasar luego a Seccion B.